In [ ]:

import os
import time
from sickle import Sickle
from almasru.client import SruClient, SruRecord, IzSruRecord, SruRequest
from almasru.utils import check_removable_records, analyse_records
from almasru import config_log
from lxml import etree


# Config logs
config_log()

# Alma

In [ ]:

# SRU
base_url = 'https://eu03-psb.alma.exlibrisgroup.com/view/sru/41SLSP_HPH'
output_dir = 'test-alma'
os.makedirs(output_dir, exist_ok=True)

# Create SRU client
client = SruClient(base_url=base_url)

# Define your query and record limit
query = 'alma.=' # HIER ERGÄNZEN

# Fetch records
sru_request = client.fetch_records(query=query, limit=50)

# Save each record as XML
for record in sru_request.records:
    mms_id = record.get_mms_id() or 'unknown'
    file_name = f"sru_{mms_id}.xml"
    file_path = os.path.join(output_dir, file_name)
    xml_bytes = etree.tostring(record.data, pretty_print=True, encoding='utf-8')
    with open(file_path, 'wb') as f:
        f.write(xml_bytes)
    print(f"Saved record as {file_name}")

print("SRU harvesting completed successfully.")

# DSpace

In [ ]:
# Define the OAI-PMH endpoint
url = 'https://demo.dspace.org/server/oai/request'
output_dir = 'test-dspace'
metadata_prefix = '' # HIER ERGÄNZEN

# Create a Sickle instance
sickle = Sickle(url)
os.makedirs(output_dir, exist_ok=True)

# Harvest records
print("Starting to harvest records...")
records = sickle.ListRecords(metadataPrefix=metadata_prefix)

max_records = 100
count = 0

for record in records:
    if count >= max_records:
        break
    timestamp = int(time.time())
    unique_id = record.header.identifier.split(':')[-1]
    sanitized_unique_id = unique_id.replace('/', '_')
    file_name = f"{timestamp}_DSPACE_OAI_{sanitized_unique_id}.xml"
    file_path = os.path.join(output_dir, file_name)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(record.raw)
    print(f"Saved record as {file_name}")
    count += 1

print("Harvesting completed successfully.")

# Backup

# Alma OAI [Backup]

In [ ]:

# OAI
url = 'https://slsp-hph-psb.alma.exlibrisgroup.com/view/oai/41SLSP_HPH/request'
output_dir = 'test-alma'
metadata_prefix = ''
set_spec = 'SLSP_FHGR'

# Create a Sickle instance
sickle = Sickle(url)
os.makedirs(output_dir, exist_ok=True)

# Harvest records
print("Starting to harvest records...")

records = sickle.ListRecords(metadataPrefix=metadata_prefix, set=set_spec)

for record_count, record in enumerate(records, start=1):
    timestamp = int(time.time())
    unique_id = record.header.identifier.split(':')[-1]
    # Create the file name using the timestamp and unique identifier
    file_name = f"{timestamp}_ALMA_OAI_TEST_{unique_id}.xml"
    file_path = os.path.join(output_dir, file_name)
    # Write the record to the file
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(record.raw)
    print(f"Saved record as {file_name}")

print("Harvesting completed successfully.")

# ArchiveSpace [Backup]

In [ ]:

# OAI Spezifika
url = 'https://sandbox.archivesspace.org/oai'
output_dir = 'test-archivesspace'
metadata_prefix='oai_dc'

# Create a Sickle instance
sickle = Sickle(url)
os.makedirs(output_dir, exist_ok=True)


# Harvest records
print("Starting to harvest records...")
records = sickle.ListRecords(metadataPrefix=metadata_prefix)

for record in records:
    timestamp = int(time.time())
    unique_id = record.header.identifier.split(':')[-1]
    # Sanitize the unique_id to remove or replace invalid characters
    sanitized_unique_id = unique_id.replace('/', '_')
    file_name = f"{timestamp}_ARCHIVESSPACE_OAI_{sanitized_unique_id}.xml"
    file_path = os.path.join(output_dir, file_name)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(record.raw)
    print(f"Saved record as {file_name}")

print("Harvesting completed successfully.")

# Koha [Backup]

In [ ]:
# Define the OAI-PMH endpoint
url = 'https://koha.adminkuhn.ch/cgi-bin/koha/oai.pl'

# Create a Sickle instance
sickle = Sickle(url)

# Harvest records with the specified metadata prefix
records = sickle.ListRecords(metadataPrefix='marc21')

# Create the output directory if it doesn't exist
output_dir = 'test-koha'
os.makedirs(output_dir, exist_ok=True)

# Save each record to a separate file with an improved naming scheme
for i, record in enumerate(records):
    timestamp = int(time.time())
    unique_id = record.header.identifier.split(':')[-1]
    file_name = f"{timestamp}_KOHA_OAI_TEST_{unique_id}.xml"
    file_path = os.path.join(output_dir, file_name)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(record.raw)